### Private local LLM model inference serving w/ Ollama
David C. Sisk, 2026-03-12

Simple examples of inference done against an Ollama server in the same private network

Why would we do this?  We are running the LLM's in this example locally (meaning within my private network, but that could also mean locally on workstation or laptop too).  The point is this: No private data is sent to any external API, it's kept internal and private.

In [10]:
# ollama needs to be installed and running locally or on a private host. See https://ollama.com/docs/quick-start 
# Ollama pulls needed:
# !ollama pull nomic-embed-text #(good smaller embedding model, can be used with the /embed endpoint in the Ollama API)
# !ollama pull codellama:13b #(or ollama pull codellama:7b for a smaller model)
# !ollama pull dolphin3:8b #(descendent of llama3:8b, with some additional fine-tuning for instruction-following tasks)


In [ ]:
# No specific python installs necessary...this uses REST API calls, so requests and json are the only imports needed.
import requests
import json

# Set up the Ollama API endpoint...I usually run it on a host in my lab with GPU's
# You could run it on a private AWS EC2 instance just as easily
ollama_url = "http://localhost:11434/api/generate"
ollama_embeddings_url = "http://localhost:11434/api/embeddings"

#ollama_url = "http://ollamaserver_nuc1:11434/api/generate"
#ollama_embeddings_url = "http://ollamaserver_nuc1:11434/api/embeddings"



Let's try retrieving a simple vector embedding from an LLM:

In [11]:
# Example 1: Vector embeddings using nomic-embed-text model
model = "nomic-embed-text"

# Text to embed
text = "Ollama makes private LLM inference serving simple!"

# Build the payload for the nomic-embed-text model
payload = {"model": model, "prompt": text}

# Post to the embeddings endpoint
response = requests.post(ollama_embeddings_url, json=payload)
result = response.json()

vector = result["embedding"]

print(f"Text: '{text}'")
print(f"Model: nomic-embed-text")
print(f"Embedding dimensions: {len(vector)}")
print(f"\nFirst 10 values of vector embedding:")
print(vector[:10])

Text: 'Ollama makes private LLM inference serving simple!'
Model: nomic-embed-text
Embedding dimensions: 768

First 10 values of vector embedding:
[0.3361373543739319, 0.9513059854507446, -3.78788161277771, -1.7929649353027344, 1.4137190580368042, -0.38124880194664, 0.7114037871360779, 0.1554303616285324, -0.4551856219768524, -0.5658385157585144]


Vector embeddings are meaningful numbers that try to capture what the text means in a semantic sense...these meaningful numbers can then be used in similarity comparison, clustering, anomaly detection, and other downstream processes.

Let's try asking an LLM trained on coding for an extendeded description of a Graph API permission:

In [7]:
# Example 2: Graph API permission description using codellama:13b

#model = "codellama:13b"
model = "codellama:7b"

permission = "EntitlementManagement.ReadWrite.All"

prompt = f"""You are an expert in Microsoft Graph API security and permissions.
Provide an extended description of the following Microsoft Graph API permission: {permission}

Include:
- What the permission allows an application to do
- The types of resources it can access or modify
- Security implications and risks of granting this permission
- Common use cases where this permission is required
- Whether it requires admin consent"""

payload = {"model": model, "prompt": prompt, "stream": False}

response = requests.post(ollama_url, json=payload)
result = response.json()

print(f"Graph API Permission: '{permission}'")
print(f"Model: {result['model']}")
print(f"\nLLM Response:{result['response']}")

Graph API Permission: 'EntitlementManagement.ReadWrite.All'
Model: codellama:7b

LLM Response:
The `EntitlementManagement.ReadWrite.All` permission allows an application to read and modify all entitlement management-related resources in your organization, including users, groups, and access packages. This permission also allows the application to manage the assignment of entitlements to users and groups, as well as to manage the lifecycle of access packages.

This permission is typically granted to applications that need to manage entitlements for users and groups, such as entitlement management tools, identity and access management (IAM) systems, and enterprise applications that require access to specific resources.

Granting this permission can potentially expose your organization's entitlement management system to security risks and vulnerabilities, such as:

* Data breaches: By granting this permission, an application may have access to sensitive information about your organization

This one takes a while for the model to respond.  However, it's a large model, it's a complex question, and the answer is also fairly complex. The model may be a bit slow, but by comparison, this would take a human being considerably longer to research, comprehend, and then word in an easily understandable way. With sufficient guard-rails, this series of models can also suggest scores that represent the level of privilege a particular permission contains. Also, note that this example is simplified for clarity...in a production application, you would want to establish sufficient guidance and guard-rails to prevent the model from making up incorrect information where it doesn't have the necessary knowledge trained-in.

Let's try comparing to pieces text to get a score reflecting how well they match:

In [8]:
# Example 3: Comparing & scoring text

model = "dolphin3:8b" # Model is descended from llama3:8b, but with some additional fine-tuning for instruction-following tasks.

# Strings to compare
string1 = "David C. Sisk"
string2 = "david christopher sisk"

# Create a prompt for the LLM to score the matching level
prompt = f"""Rate the similarity between these two names on a scale of 0-100, where 0 is completely different and 100 is identical.
Name 1: {string1}
Name 2: {string2}
Provide only a numeric score and a brief explanation."""

# Send request to Ollama...use Dolphin3:8b, which is descended from llama3:8b
payload = {"model": model, "prompt": prompt, "stream": False}

response = requests.post(ollama_url, json=payload)
result = response.json()

print(f"Comparing: '{string1}' vs '{string2}'")
print(f"\nLLM Response:\n{result['response']}")

Comparing: 'David C. Sisk' vs 'david christopher sisk'

LLM Response:
Score: 98

Explanation: The names David C. Sisk and david christopher sisk are very similar, as they only differ in the capitalization and spacing of the letters.


The score is good, but the explanation isn't 100% accurate. Regardless, this is much better than requiring a human being to do this boring repetitive task! 

So...what's the best way to secure private data sent on external API calls?  <b>Answer: Don't send any private data on external API calls.</b> Not sending private data external in the first place is how regulated industries (healthcare, insurance, financials, government, etc) prevent data leakage and other exposures with 100% certainty (at least from the application stack).